# 🪙 Path of Exile 2 Economy Data Explorer

Note: This is generated with ChatGPT, to show an example of use.

This notebook downloads and caches market data from [poe.ninja](https://poe.ninja/poe2/economy/) using the `pyoe2_craftpath` library, parses it into an economy object, and provides tools to explore and visualize the results.

In [1]:
import os
import time
import json
import requests
import pandas as pd
from pprint import pprint
import pyoe2_craftpath as pc

## Configuration

In [2]:
MARKET_MAP = {
    "./cache/pn_abyss.json": "https://poe.ninja/poe2/api/economy/exchange/current/overview?league=Rise of the Abyssal&type=Abyss",
    "./cache/pn_currency.json": "https://poe.ninja/poe2/api/economy/exchange/current/overview?league=Rise of the Abyssal&type=Currency",
    "./cache/pn_essences.json": "https://poe.ninja/poe2/api/economy/exchange/current/overview?league=Rise of the Abyssal&type=Essences",
    "./cache/pn_ritual.json": "https://poe.ninja/poe2/api/economy/exchange/current/overview?league=Rise of the Abyssal&type=Ritual"
}

CACHE_TTL_IN_SECONDS = 60 * 60  # 1 hour

os.makedirs("./cache", exist_ok=True)

## Fetch or Load Cached Data With Inbuilt Function

In [3]:
raw_fetched_responses = pc.retrieve_jsons_from_urls_with_cache(
    cache_url_map=MARKET_MAP,
    max_cache_duration_in_sec=CACHE_TTL_IN_SECONDS
)

print(f"Loaded {len(raw_fetched_responses)} market data files.")

Loaded 4 market data files.
2025-10-31T21:56:29.110925Z  INFO retrieve_jsons_from_urls_with_cache: Downloading fresh data for ./cache/pn_currency.json...
2025-10-31T21:56:29.663546Z  INFO retrieve_jsons_from_urls_with_cache: Downloading fresh data for ./cache/pn_abyss.json...
2025-10-31T21:56:31.674858Z  INFO retrieve_jsons_from_urls_with_cache: Downloading fresh data for ./cache/pn_ritual.json...
2025-10-31T21:56:32.506656Z  INFO retrieve_jsons_from_urls_with_cache: Downloading fresh data for ./cache/pn_essences.json...


## Parse the Economy Data

In [4]:
economy = pc.parse_economy_from_jsons(raw_fetched_responses)

## Basic Validations

In [5]:
test_currency = economy.cache_market_prices.get(pc.ItemName("Perfect Orb of Transmutation"))
test_ritual = economy.cache_market_prices.get(pc.ItemName("Omen of the Blackblooded"))
test_essence = economy.cache_market_prices.get(pc.ItemName("Perfect Essence of Ruin"))
test_abyss = economy.cache_market_prices.get(pc.ItemName("Kulemak's Invitation"))

assert test_currency is not None
assert test_ritual is not None
assert test_essence is not None
assert test_abyss is not None

print("✅ Economy data loaded and validated successfully!")

✅ Economy data loaded and validated successfully!


## Example Conversions

In [6]:
print("Abyss Item Divine Value:", economy.currency_convert(test_abyss, pc.PriceKind.Divine))
print("Abyss Item Exalted Value:", economy.currency_convert(test_abyss, pc.PriceKind.Exalted))
print("Abyss Item Chaos Value:", economy.currency_convert(test_abyss, pc.PriceKind.Chaos))

Abyss Item Divine Value: 0.03733
Abyss Item Exalted Value: 61.669160000000005
Abyss Item Chaos Value: 1.1811212


## Convert Market Prices to DataFrame

In [ ]:
data = [
    {
        "Item": str(item_name.raw_value), # (move out of wrapper to str)
        "Value (Divine)": price.get_divine_value(),
        "Value (Chaos)": economy.currency_convert(price, pc.PriceKind.Chaos),
        "Value (Exalted)": economy.currency_convert(price, pc.PriceKind.Exalted),
    }
    for item_name, price in economy.cache_market_prices.items()
]

df = pd.DataFrame(data).sort_values(by="Value (Divine)", ascending=False)
df.head(20)

,Item,Value (Divine),Value (Chaos),Value (Exalted)
131,Mirror of Kalandra,1824.0000,57711.360000,3.013248e+06
113,Hinekora's Lock,662.5000,20961.500000,1.094450e+06
140,Ancient Collarbone,7.5000,237.300000,1.239000e+04
148,Omen of Sinistral Annulment,6.7400,213.253600,1.113448e+04
93,Omen of Dextral Annulment,6.2000,196.168000,1.024240e+04
25,Omen of Chance,4.7200,149.340800,7.797440e+03
75,Omen of Sinistral Erasure,4.3600,137.950400,7.202720e+03
141,Omen of Light,4.1900,132.571600,6.921880e+03
89,Ancient Jawbone,4.1300,130.673200,6.822760e+03
135,Omen of Dextral Erasure,4.0500,128.142000,6.690600e+03


## Plot Top 20 Most Valuable Items (Divine Value)

In [8]:
import matplotlib.pyplot as plt

top20 = df.head(20)
plt.figure(figsize=(10, 6))
plt.barh(top20["Item"], top20["Value (Divine)"])
plt.gca().invert_yaxis()
plt.xlabel("Divine Value")
plt.title("Top 20 Most Valuable Items (Divine Value)")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## Search Utility

In [ ]:
def search_item(name_fragment: str):
    """Search for items by partial name."""
    return df[df["Item"].str.contains(name_fragment, case=False, na=False)]

# Example:
search_item("Divine")

## Interactive Exploration
You can re-run the search cell with different phrases like `"Essence"`, `"Omen"`, `"Orb"`, etc.